In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
#load dataset
df=pd.read_csv("Churn_Modelling.csv")

In [ ]:
df

In [ ]:
#droping irrelevent Data
df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
#label encoding gender and geography
label_encoder_gender=LabelEncoder()
df['Gender']=label_encoder_gender.fit_transform(df['Gender'])

#ONEHOT encoding geography
encoder=OneHotEncoder()
encoded=encoder.fit_transform(df[['Geography']]).toarray()
encoder_df=pd.DataFrame(encoded,columns=encoder.get_feature_names_out())
df=pd.concat([df,encoder_df],axis=1)
df.drop(['Geography'],axis=1,inplace=True)


In [ ]:
df.head()

In [ ]:
X=df.drop(['Exited'],axis=1)
y=df['Exited']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)


In [ ]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping
import datetime


In [ ]:
#build model
model=Sequential(
    [Dense(64,activation='relu',input_shape=(X_train.shape[1],)),   ###HL1 conected with input layer
    Dense(32,activation='relu'),#hl2
    Dense(1,activation="sigmoid")#output layer
    ]
)

In [ ]:
model.summary()

In [ ]:
#compile model
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

In [ ]:
#setup tensorboard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
## Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [ ]:
### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
## Load Tensorboard Extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir regressionlogs/fit/20260106-151608